# Enhanced py-ard: Donor-Recipient Compatibility Analysis

This notebook demonstrates enhanced donor-recipient compatibility analysis functionality developed for py-ard, powered by the [HLAtools](https://github.com/sjmack/HLAtools) R package via rpy2.

## Features Demonstrated
- HLAtools R package integration via `pyard.alignment_bridge` for authoritative alignment data
- Real amino acid sequence analysis with per-residue IMGT position numbering
- Accurate leader/mature protein boundary classification using IMGT positions directly (negative = leader, positive = mature)
- Donor-recipient genotype compatibility assessment
- Detailed mismatch analysis with biological context
- CSV export with IMGT position and sequence region annotations
- Multi-locus support (A, B, C, DRB1, DPB1, DQB1, and more)

## Prerequisites
- **R** (>= 3.6) with the **HLAtools** package installed
- **rpy2** and **pandas** Python packages (`pip install rpy2 pandas`)
- py-ard project root on `sys.path` (handled automatically by the first code cell)

If HLAtools/rpy2 are not available, the notebook falls back to direct IMGT text file parsing.

## Leader/Mature Boundary Resolution

Previous versions of this notebook used interpolation to approximate IMGT position numbers from the raw IMGT alignment text files. This led to inaccurate leader/mature boundary determination -- for example, DPB1 showed 91 leader positions instead of the correct 29.

This has been resolved by integrating HLAtools, which provides IMGT position numbers as data frame column headers. The position mapping is now exact: negative positions are leader peptide, positive positions are mature protein, with no interpolation needed. For DPB1, the boundary falls cleanly at IMGT position -1 (leader) to position 1 (mature), with 29 leader and 244 mature positions.

## Enhanced Protein Alignment Functions

In [1]:
# Ensure project root is on sys.path so pyard is importable
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Try to use HLAtools bridge (preferred) with fallback to direct IMGT parsing
USE_HLATOOLS = False
bridge = None

try:
    from pyard.alignment_bridge import HLAToolsBridge

    bridge = HLAToolsBridge()
    if bridge.is_available:
        USE_HLATOOLS = True
        print("HLAtools bridge is available - will use R-based alignment data")
        print("Note: First call to get_alignment() will load data (may take several minutes)")
    else:
        print("HLAtools not available - falling back to direct IMGT parsing")
except ImportError:
    print("pyard alignment_bridge not installed - falling back to direct IMGT parsing")

print(f"Using HLAtools: {USE_HLATOOLS}")

HLAtools bridge is available - will use R-based alignment data
Note: First call to get_alignment() will load data (may take several minutes)
Using HLAtools: True


In [2]:
# Import required modules
from urllib.request import urlopen
from typing import Dict
import sys
import re

def load_protein_alignment_enhanced(locus: str) -> Dict[str, str]:
    """
    Load protein alignment for a specific locus from IMGT/HLA GitHub repository.
    Enhanced version that handles multi-block sequences correctly.
    """
    # Handle special DRB cases
    if locus in ["DRB1", "DRB3", "DRB4", "DRB5"]:
        file_locus = "DRB"
    else:
        file_locus = locus
    
    try:
        url = f"https://raw.githubusercontent.com/ANHIG/IMGTHLA/Latest/alignments/{file_locus}_prot.txt"
        print(f"Loading alignment data from: {url}")
        
        response = urlopen(url)
        lines = [line.decode("utf-8").rstrip() for line in response]
        
        # Find the data start (first actual allele sequence)
        data_start_index = None
        for i, line in enumerate(lines):
            stripped_line = line.strip()
            if (stripped_line.startswith(f'{locus}*') and 
                not stripped_line.startswith('Prot') and
                len(stripped_line.split(None, 1)) > 1):
                data_start_index = i
                break
        
        if data_start_index is None:
            print(f"Could not find data start in alignment for {locus}")
            return {}
        
        print(f"Found data starting at line {data_start_index}")
        
        # Collect all sequence blocks for each allele
        all_sequence_blocks = {}
        
        for line_num, line in enumerate(lines[data_start_index:], data_start_index):
            stripped_line = line.strip()
            
            if not stripped_line:
                continue
                
            if stripped_line.startswith(f'{locus}*'):
                parts = stripped_line.split(None, 1)
                if len(parts) >= 2:
                    allele_name = parts[0]
                    sequence_data = parts[1]
                    
                    # Extract amino acid sequence (remove spaces, dots, keep letters, dashes, asterisks)
                    sequence_block = ''.join(char for char in sequence_data 
                                           if char.isalpha() or char == '-' or char == '*')
                    
                    if allele_name not in all_sequence_blocks:
                        all_sequence_blocks[allele_name] = []
                    
                    all_sequence_blocks[allele_name].append(sequence_block)
        
        print(f"Collected sequence blocks for {len(all_sequence_blocks)} alleles")
        
        # Find reference sequence (first allele with actual amino acids, not mostly dashes)
        reference_allele = None
        reference_sequence = None
        
        for allele_name, blocks in all_sequence_blocks.items():
            full_sequence = ''.join(blocks)
            dash_count = full_sequence.count('-')
            if dash_count <= len(full_sequence) * 0.5:  # Less than 50% dashes
                reference_allele = allele_name
                reference_sequence = full_sequence
                print(f"Using {allele_name} as reference sequence ({len(reference_sequence)} AA, {dash_count} dashes)")
                break
        
        if reference_sequence is None:
            print(f"Could not find reference sequence for {locus}")
            return {}
        
        # Resolve all sequences using the reference
        alignment_dict = {}
        for allele_name, blocks in all_sequence_blocks.items():
            full_sequence = ''.join(blocks)
            
            if allele_name == reference_allele:
                resolved_sequence = full_sequence
            else:
                # Resolve dashes using reference
                resolved_sequence = resolve_sequence_with_reference(full_sequence, reference_sequence)
            
            alignment_dict[allele_name] = resolved_sequence
        
        print(f"Successfully loaded {len(alignment_dict)} resolved sequences")
        return alignment_dict
        
    except Exception as e:
        print(f"Error parsing alignment for {locus}: {e}")
        return {}

def resolve_sequence_with_reference(sequence: str, reference: str) -> str:
    """
    Convert dash notation to actual amino acid sequence using reference.
    """
    if len(sequence) != len(reference):
        min_len = min(len(sequence), len(reference))
        sequence = sequence[:min_len]
        reference = reference[:min_len]
    
    result = ""
    for i, char in enumerate(sequence):
        if char == '-':
            result += reference[i]
        elif char == '*':
            result += '*'
        else:
            result += char
            
    return result

print("Enhanced protein alignment functions loaded")

Enhanced protein alignment functions loaded


## Test 1: Load and Examine Protein Alignment Data

In [3]:
# Test loading DPB1 alignment data
print("Testing Enhanced Protein Alignment Loading")
print("=" * 50)

if USE_HLATOOLS:
    print("Using HLAtools bridge for alignment data")
    dpb1_alignment = bridge.get_alignment("DPB1")
else:
    dpb1_alignment = load_protein_alignment_enhanced("DPB1")

if dpb1_alignment:
    print(f"\nSuccessfully loaded {len(dpb1_alignment)} DPB1 protein sequences")
    
    # Show some example sequences
    print("\nExample sequences:")
    count = 0
    for allele, sequence in dpb1_alignment.items():
        if count < 5:  # Show first 5
            print(f"  {allele}: {sequence[:60]}... (length: {len(sequence)})")
            count += 1
    
    # Test specific alleles we'll use for comparison
    test_alleles = ["DPB1*04:01:01:01", "DPB1*04:02:01:01", "DPB1*05:01:01:01"]
    
    print(f"\nChecking for specific test alleles:")
    available_alleles = []
    for allele in test_alleles:
        if allele in dpb1_alignment:
            available_alleles.append(allele)
            print(f"  {allele}: Available (length {len(dpb1_alignment[allele])})")
        else:
            # Try to find similar alleles
            similar = [a for a in dpb1_alignment.keys() if a.startswith(allele.split(':')[0])]
            if similar:
                available_alleles.append(similar[0])
                print(f"  {allele}: Not found, using {similar[0]} instead")
            else:
                print(f"  {allele}: Not available")
    
    print(f"\nWill use these alleles for testing: {available_alleles[:3]}")
    
else:
    print("Failed to load DPB1 alignment data")

Testing Enhanced Protein Alignment Loading
Using HLAtools bridge for alignment data



Successfully loaded 2889 DPB1 protein sequences

Example sequences:
  DPB1*01:01:01:01: MMVLQVSAAPRTVALTALLMVLLTSVVQGRATPENYVYQGRQECYAFNGTQRFLERYIYN... (length: 258)
  DPB1*01:01:01:02: MMVLQVSAAPRTVALTALLMVLLTSVVQGRATPENYVYQGRQECYAFNGTQRFLERYIYN... (length: 258)
  DPB1*01:01:01:03: MMVLQVSAAPRTVALTALLMVLLTSVVQGRATPENYVYQGRQECYAFNGTQRFLERYIYN... (length: 258)
  DPB1*01:01:01:04: MMVLQVSAAPRTVALTALLMVLLTSVVQGRATPENYVYQGRQECYAFNGTQRFLERYIYN... (length: 258)
  DPB1*01:01:01:05: MMVLQVSAAPRTVALTALLMVLLTSVVQGRATPENYVYQGRQECYAFNGTQRFLERYIYN... (length: 258)

Checking for specific test alleles:
  DPB1*04:01:01:01: Available (length 258)
  DPB1*04:02:01:01: Available (length 258)
  DPB1*05:01:01:01: Available (length 258)

Will use these alleles for testing: ['DPB1*04:01:01:01', 'DPB1*04:02:01:01', 'DPB1*05:01:01:01']


## Core Analysis Functions

In [4]:
# Enhanced sequence comparison functions
def get_amino_acid_sequence_enhanced(allele: str, alignment_cache: dict = None):
    """
    Get amino acid sequence for an allele using enhanced protein alignment data.
    Uses HLAtools bridge when available, falls back to direct IMGT parsing.
    """
    if not allele or '*' not in allele:
        return None, "error"
    
    locus = allele.split('*')[0]
    
    # Try HLAtools bridge first
    if USE_HLATOOLS and bridge is not None:
        seq = bridge.get_sequence(allele)
        if seq is not None:
            return seq, "hlatools"
    
    # Fallback: use provided cache or load fresh from IMGT
    if alignment_cache and locus in alignment_cache:
        alignment = alignment_cache[locus]
    else:
        alignment = load_protein_alignment_enhanced(locus)
        if alignment_cache is not None:
            alignment_cache[locus] = alignment
    
    if not alignment:
        return None, "error"
    
    # Try exact match first
    if allele in alignment:
        return alignment[allele], "protein_alignment"
    
    # Try to find best match by reducing allele name
    allele_parts = allele.split(':')
    for i in range(len(allele_parts), 1, -1):
        reduced_allele = ':'.join(allele_parts[:i])
        for full_allele in alignment.keys():
            if full_allele.startswith(reduced_allele):
                return alignment[full_allele], "protein_alignment"
    
    return None, "error"

def compare_single_alleles(donor_allele: str, recipient_allele: str, alignment_cache: dict = None) -> dict:
    """
    Compare two specific alleles for amino acid mismatches using enhanced sequence data.
    """
    # Get amino acid sequences using enhanced method
    try:
        donor_seq, donor_method = get_amino_acid_sequence_enhanced(donor_allele, alignment_cache)
        recipient_seq, recipient_method = get_amino_acid_sequence_enhanced(recipient_allele, alignment_cache)
    except Exception as e:
        return {"error": f"Could not retrieve sequences: {str(e)}"}
    
    if not donor_seq or not recipient_seq:
        return {
            "error": f"Could not retrieve sequences for {donor_allele} and/or {recipient_allele}",
            "donor_allele": donor_allele,
            "recipient_allele": recipient_allele,
            "mismatch_count": None
        }
    
    # Compare sequences
    mismatches = []
    for i, (donor_aa, recipient_aa) in enumerate(zip(donor_seq, recipient_seq)):
        if donor_aa != recipient_aa:
            mismatches.append({
                "position": i + 1,
                "donor_aa": donor_aa,
                "recipient_aa": recipient_aa,
                "change": f"{donor_aa}>{recipient_aa}"
            })
    
    return {
        "donor_allele": donor_allele,
        "recipient_allele": recipient_allele,
        "mismatch_count": len(mismatches),
        "mismatch_positions": [m["position"] for m in mismatches],
        "substitutions": [m["change"] for m in mismatches],
        "mismatches": mismatches,
        "donor_sequence_length": len(donor_seq),
        "recipient_sequence_length": len(recipient_seq),
        "analysis_method": f"{donor_method}/{recipient_method}"
    }

print("Core analysis functions loaded")

Core analysis functions loaded


## Test 2: Single Allele Comparison

In [5]:
# Test single allele comparison with real amino acid sequences
print("Testing Single Allele Comparison")
print("=" * 40)

# Create alignment cache to avoid reloading
alignment_cache = {}

# Test DPB1*04:01 vs DPB1*04:02 (known to have differences)
donor_allele = "DPB1*04:01:01:01"
recipient_allele = "DPB1*04:02:01:01"

print(f"Comparing {donor_allele} vs {recipient_allele}")

result = compare_single_alleles(donor_allele, recipient_allele, alignment_cache)

if result.get("error"):
    print(f"Error: {result['error']}")
    
    # Try with available alleles from our earlier test
    if dpb1_alignment:
        available = list(dpb1_alignment.keys())
        if len(available) >= 2:
            donor_allele = available[0]
            recipient_allele = available[1]
            print(f"\nTrying with available alleles: {donor_allele} vs {recipient_allele}")
            result = compare_single_alleles(donor_allele, recipient_allele, alignment_cache)

if not result.get("error"):
    print(f"\nComparison Results:")
    print(f"  Donor: {result['donor_allele']} (length: {result['donor_sequence_length']})")
    print(f"  Recipient: {result['recipient_allele']} (length: {result['recipient_sequence_length']})")
    print(f"  Mismatches: {result['mismatch_count']}")
    print(f"  Analysis method: {result['analysis_method']}")
    
    if result['mismatch_count'] > 0:
        print(f"  Mismatch positions: {result['mismatch_positions'][:10]}{'...' if len(result['mismatch_positions']) > 10 else ''}")
        print(f"  Substitutions: {result['substitutions'][:10]}{'...' if len(result['substitutions']) > 10 else ''}")
        
        print(f"\nFirst 5 detailed mismatches:")
        for i, mismatch in enumerate(result['mismatches'][:5]):
            print(f"    Position {mismatch['position']}: {mismatch['change']}")
    else:
        print(f"  No mismatches found - sequences are identical!")
else:
    print("Could not perform comparison")

Testing Single Allele Comparison
Comparing DPB1*04:01:01:01 vs DPB1*04:02:01:01

Comparison Results:
  Donor: DPB1*04:01:01:01 (length: 258)
  Recipient: DPB1*04:02:01:01 (length: 258)
  Mismatches: 4
  Analysis method: hlatools/hlatools
  Mismatch positions: [65, 84, 85, 207]
  Substitutions: ['A>V', 'A>D', 'A>E', 'L>M']

First 5 detailed mismatches:
    Position 65: A>V
    Position 84: A>D
    Position 85: A>E
    Position 207: L>M


## Position Mapping Functions

In [6]:
# Position mapping functions for leader/mature classification
def get_detailed_position_mapping(locus: str) -> dict:
    """Correctly map sequence indices to IMGT positions across all blocks."""
    from urllib.request import urlopen
    import re
    
    file_locus = "DRB" if locus in ["DRB1", "DRB3", "DRB4", "DRB5"] else locus
    
    try:
        url = f"https://raw.githubusercontent.com/ANHIG/IMGTHLA/Latest/alignments/{file_locus}_prot.txt"
        response = urlopen(url)
        lines = [line.decode('utf-8').rstrip() for line in response]
        
        position_mapping = {}
        sequence_index = 0
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            if "Prot" in line and re.search(r'-?\d+', line):
                if i + 1 < len(lines) and "|" in lines[i + 1]:
                    marker_line = lines[i + 1]
                    imgt_positions = [int(n) for n in re.findall(r'-?\d+', line)]
                    
                    # Find the sequence block that follows
                    block_start = i + 2
                    first_seq_line = None
                    for seq_line_idx in range(block_start, min(block_start + 10, len(lines))):
                        if lines[seq_line_idx].strip().startswith(f'{locus}*'):
                            first_seq_line = lines[seq_line_idx]
                            break
                    
                    if first_seq_line:
                        parts = first_seq_line.split(None, 1)
                        if len(parts) >= 2:
                            seq_data = parts[1]
                            aa_count = sum(1 for c in seq_data if c.isalpha())
                            
                            # Map positions in this block
                            if len(imgt_positions) >= 2:
                                start_imgt = imgt_positions[0]
                                end_imgt = imgt_positions[-1]
                                for j in range(aa_count):
                                    if aa_count > 1:
                                        imgt_pos = start_imgt + (end_imgt - start_imgt) * j / (aa_count - 1)
                                    else:
                                        imgt_pos = start_imgt
                                    position_mapping[sequence_index + j] = int(imgt_pos)
                            elif len(imgt_positions) == 1:
                                imgt_pos = imgt_positions[0]
                                for j in range(aa_count):
                                    position_mapping[sequence_index + j] = imgt_pos + j
                            
                            sequence_index += aa_count
            i += 1
        
        return position_mapping
        
    except Exception as e:
        print(f"Error creating position mapping for {locus}: {e}")
        return {}

def create_position_index_mapping(
    first_sequence: str, allele_name_length: int, 
    marker_positions: list, position_numbers: list
) -> dict:
    """
    Create mapping from sequence index to IMGT position using interpolation.
    """
    index_to_imgt = {}
    sequence_index = 0
    
    for char_pos, char in enumerate(first_sequence):
        if char.isalpha():
            # Calculate character position in full line
            adjusted_char_pos = char_pos + allele_name_length
            
            # Interpolate IMGT position
            imgt_pos = interpolate_imgt_position(
                adjusted_char_pos, marker_positions, position_numbers
            )
            
            if imgt_pos is not None:
                index_to_imgt[sequence_index] = imgt_pos
            
            sequence_index += 1
    
    print(f"Created position mapping for {len(index_to_imgt)} amino acid positions")
    return index_to_imgt

def interpolate_imgt_position(
    char_pos: int, marker_positions: list, position_numbers: list
) -> int:
    """
    Interpolate IMGT position for a character position between known markers.
    """
    # Find surrounding markers
    left_marker_idx = None
    right_marker_idx = None
    
    for i, marker_pos in enumerate(marker_positions):
        if marker_pos <= char_pos:
            left_marker_idx = i
        elif marker_pos > char_pos and right_marker_idx is None:
            right_marker_idx = i
            break
    
    if left_marker_idx is None or right_marker_idx is None:
        return None
    
    # Interpolate position
    left_pos = marker_positions[left_marker_idx]
    right_pos = marker_positions[right_marker_idx]
    left_imgt = position_numbers[left_marker_idx]
    right_imgt = position_numbers[right_marker_idx]
    
    # Calculate offset and interpolate
    offset = char_pos - left_pos
    total_span = right_pos - left_pos
    imgt_span = right_imgt - left_imgt
    
    if total_span == 0:
        return left_imgt
    
    interpolated_imgt = left_imgt + int((offset / total_span) * imgt_span)
    return interpolated_imgt

print("Position mapping functions loaded")

Position mapping functions loaded


## Test 3: Position Mapping and Leader/Mature Classification

In [7]:
# Test position mapping for leader/mature classification
print("Testing Position Mapping for Leader/Mature Classification")
print("=" * 60)

# Test with DPB1
locus = "DPB1"

if USE_HLATOOLS:
    print("Using HLAtools bridge for position mapping (IMGT positions from column headers)")
    position_mapping = bridge.get_position_mapping(locus)
else:
    position_mapping = get_detailed_position_mapping(locus)

if position_mapping:
    print(f"\nSuccessfully created position mapping with {len(position_mapping)} positions")
    
    # Show some example mappings
    print("\nExample sequence index -> IMGT position mappings:")
    example_positions = list(position_mapping.items())[:20]
    for seq_idx, imgt_pos in example_positions:
        region = "Leader" if imgt_pos < 1 else "Mature"
        print(f"  Sequence position {seq_idx+1:3d} -> IMGT position {imgt_pos:3d} ({region})")
    
    # Count leader vs mature positions
    leader_positions = [pos for pos in position_mapping.values() if pos < 1]
    mature_positions = [pos for pos in position_mapping.values() if pos >= 1]
    
    print(f"\nSequence region summary:")
    print(f"  Leader sequence positions: {len(leader_positions)}")
    print(f"  Mature protein positions: {len(mature_positions)}")
    print(f"  Total mapped positions: {len(position_mapping)}")
    
    # Show the boundary
    boundary_positions = [(idx, pos) for idx, pos in position_mapping.items() if -5 <= pos <= 5]
    if boundary_positions:
        print(f"\nLeader/Mature boundary region:")
        for seq_idx, imgt_pos in sorted(boundary_positions):
            region = "Leader" if imgt_pos < 1 else "Mature"
            print(f"  Sequence position {seq_idx+1:3d} -> IMGT position {imgt_pos:3d} ({region})")

else:
    print("Failed to create position mapping")

Testing Position Mapping for Leader/Mature Classification
Using HLAtools bridge for position mapping (IMGT positions from column headers)



Successfully created position mapping with 273 positions

Example sequence index -> IMGT position mappings:
  Sequence position   1 -> IMGT position -29 (Leader)
  Sequence position   2 -> IMGT position -28 (Leader)
  Sequence position   3 -> IMGT position -27 (Leader)
  Sequence position   4 -> IMGT position -26 (Leader)
  Sequence position   5 -> IMGT position -25 (Leader)
  Sequence position   6 -> IMGT position -24 (Leader)
  Sequence position   7 -> IMGT position -23 (Leader)
  Sequence position   8 -> IMGT position -22 (Leader)
  Sequence position   9 -> IMGT position -21 (Leader)
  Sequence position  10 -> IMGT position -20 (Leader)
  Sequence position  11 -> IMGT position -19 (Leader)
  Sequence position  12 -> IMGT position -18 (Leader)
  Sequence position  13 -> IMGT position -17 (Leader)
  Sequence position  14 -> IMGT position -16 (Leader)
  Sequence position  15 -> IMGT position -15 (Leader)
  Sequence position  16 -> IMGT position -14 (Leader)
  Sequence position  17 -> 

## Enhanced Analysis Functions

In [8]:
# Enhanced genotype analysis functions
def parse_genotype_alleles(genotype: str, locus: str) -> list:
    """
    Parse genotype string to extract individual alleles.
    """
    clean_genotype = genotype.replace("HLA-", "")
    
    # Split by '/' or '+' to get individual alleles
    if "/" in clean_genotype:
        allele_parts = clean_genotype.split("/")
    elif "+" in clean_genotype:
        allele_parts = clean_genotype.split("+")
    else:
        allele_parts = [clean_genotype]
    
    alleles = []
    for part in allele_parts:
        part = part.strip()
        if not part:
            continue
            
        if part.startswith(f"{locus}*"):
            alleles.append(part)
        elif "*" not in part or not part.split("*")[0]:
            alleles.append(f"{locus}*{part}")
        else:
            alleles.append(part)
    
    return alleles

def analyze_donor_recipient_compatibility(
    donor_genotype: str, recipient_genotype: str, locus: str, alignment_cache: dict = None
) -> dict:
    """
    Analyze amino acid compatibility between donor and recipient genotypes.
    Handles only one locus at a time.
    """
    # Parse alleles
    donor_alleles = parse_genotype_alleles(donor_genotype, locus)
    recipient_alleles = parse_genotype_alleles(recipient_genotype, locus)
    
    if not donor_alleles or not recipient_alleles:
        return {"error": "Could not parse genotype alleles"}
    
    print(f"Donor alleles: {donor_alleles}")
    print(f"Recipient alleles: {recipient_alleles}")
    
    # Analyze all combinations
    combinations = []
    for donor_allele in donor_alleles:
        for recipient_allele in recipient_alleles:
            print(f"Analyzing {donor_allele} vs {recipient_allele}...")
            result = compare_single_alleles(donor_allele, recipient_allele, alignment_cache)
            combinations.append(result)
    
    # Summary statistics
    mismatch_counts = [c["mismatch_count"] for c in combinations if c["mismatch_count"] is not None]
    
    return {
        "donor_genotype": donor_genotype,
        "recipient_genotype": recipient_genotype,
        "locus": locus,
        "total_combinations": len(combinations),
        "min_mismatches": min(mismatch_counts) if mismatch_counts else None,
        "max_mismatches": max(mismatch_counts) if mismatch_counts else None,
        "combinations": combinations
    }

print("Enhanced analysis functions loaded")

Enhanced analysis functions loaded


## Test 4: Donor-Recipient Genotype Analysis

In [9]:
# Test donor-recipient genotype analysis
print("Testing Donor-Recipient Genotype Analysis")
print("=" * 50)

compatibility_result = None

# Use available alleles for realistic testing
if dpb1_alignment and len(dpb1_alignment) >= 4:
    available_alleles = list(dpb1_alignment.keys())[:4]
    
    # Use hard-coded example genotypes
    donor_genotype = "DPB1*04:01/04:02"
    recipient_genotype = "DPB1*05:01/06:01" 
    locus = "DPB1"

    print(f"Testing genotype compatibility:")
    print(f"  Donor: {donor_genotype}")
    print(f"  Recipient: {recipient_genotype}")
    print(f"  Locus: {locus}")
    
    compatibility_result = analyze_donor_recipient_compatibility(donor_genotype, recipient_genotype, locus, alignment_cache)
    
    if compatibility_result.get("error"):
        print(f"Error: {compatibility_result['error']}")
    else:
        print(f"\nAnalysis Results:")
        print(f"  Total combinations: {compatibility_result['total_combinations']}")
        print(f"  Mismatch range: {compatibility_result['min_mismatches']}-{compatibility_result['max_mismatches']}")
        
        print(f"\nIndividual Combinations:")
        for i, combo in enumerate(compatibility_result['combinations'], 1):
            if not combo.get('error'):
                donor_short = combo['donor_allele'].split('*')[1] if '*' in combo['donor_allele'] else combo['donor_allele']
                recipient_short = combo['recipient_allele'].split('*')[1] if '*' in combo['recipient_allele'] else combo['recipient_allele']
                print(f"  {i}. {donor_short} → {recipient_short}: {combo['mismatch_count']} mismatches")
                
                if combo['mismatch_count'] > 0 and combo['mismatch_count'] <= 8:
                    print(f"     Substitutions: {', '.join(combo['substitutions'][:5])}{'...' if len(combo['substitutions']) > 5 else ''}")
            else:
                print(f"  {i}. Error: {combo['error']}")
else:
    print("Insufficient alleles available for genotype testing")

Testing Donor-Recipient Genotype Analysis
Testing genotype compatibility:
  Donor: DPB1*04:01/04:02
  Recipient: DPB1*05:01/06:01
  Locus: DPB1
Donor alleles: ['DPB1*04:01', 'DPB1*04:02']
Recipient alleles: ['DPB1*05:01', 'DPB1*06:01']
Analyzing DPB1*04:01 vs DPB1*05:01...
Analyzing DPB1*04:01 vs DPB1*06:01...
Analyzing DPB1*04:02 vs DPB1*05:01...
Analyzing DPB1*04:02 vs DPB1*06:01...

Analysis Results:
  Total combinations: 4
  Mismatch range: 10-15

Individual Combinations:
  1. 04:01 → 05:01: 10 mismatches
  2. 04:01 → 06:01: 15 mismatches
  3. 04:02 → 05:01: 10 mismatches
  4. 04:02 → 06:01: 15 mismatches


## Test 5: CSV Export with Position Classification

In [10]:
# Test CSV export with leader/mature classification
def export_analysis_to_csv(analysis_result: dict, position_mapping: dict, filename: str) -> str:
    """
    Export analysis to CSV with position classification.
    """
    import csv
    
    if analysis_result.get("error"):
        return f"Error: {analysis_result['error']}"
    
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        
        # Write header
        writer.writerow([
            "Donor_Allele", 
            "Recipient_Allele", 
            "Total_Mismatches", 
            "Position", 
            "Substitution",
            "Sequence_Region",
            "IMGT_Position"
        ])
        
        # Write data
        for combo in analysis_result['combinations']:
            if combo.get("error"):
                continue
                
            donor_short = combo['donor_allele'].split('*')[1] if '*' in combo['donor_allele'] else combo['donor_allele']
            recipient_short = combo['recipient_allele'].split('*')[1] if '*' in combo['recipient_allele'] else combo['recipient_allele']
            total_mismatches = combo['mismatch_count']
            
            if total_mismatches == 0:
                writer.writerow([donor_short, recipient_short, 0, "None", "None", "None", "None"])
            else:
                for pos, sub in zip(combo['mismatch_positions'], combo['substitutions']):
                    # Determine sequence region
                    # Debug the position lookup
                    seq_index = pos - 1  # Convert to 0-based
                    imgt_position = position_mapping.get(seq_index)
                    
                    if imgt_position is not None:
                        sequence_region = "Leader" if imgt_position < 1 else "Mature"
                        imgt_pos_str = str(imgt_position)
                    else:
                        sequence_region = "Unknown"
                        imgt_pos_str = "Unknown"
                    
                    writer.writerow([
                        donor_short, 
                        recipient_short, 
                        total_mismatches, 
                        pos, 
                        sub,
                        sequence_region,
                        imgt_pos_str
                    ])
    
    return f"Results exported to {filename}"

# Test CSV export if we have analysis results
print("Testing CSV Export with Position Classification")
print("=" * 50)

if compatibility_result is not None and not compatibility_result.get('error') and position_mapping:
    csv_filename = "standalone_demo_results.csv"
    export_status = export_analysis_to_csv(compatibility_result, position_mapping, csv_filename)
    print(export_status)
    
    # Show first few lines
    try:
        with open(csv_filename, 'r') as f:
            lines = f.readlines()[:15]
        
        print(f"\nFirst 15 lines of {csv_filename}:")
        for i, line in enumerate(lines, 1):
            print(f"{i:2d}: {line.strip()}")
        
        # Count regions
        with open(csv_filename, 'r') as f:
            import csv
            reader = csv.DictReader(f)
            leader_count = 0
            mature_count = 0
            
            for row in reader:
                if row['Sequence_Region'] == 'Leader':
                    leader_count += 1
                elif row['Sequence_Region'] == 'Mature':
                    mature_count += 1
            
            print(f"\nSequence Region Summary:")
            print(f"  Leader mismatches: {leader_count}")
            print(f"  Mature mismatches: {mature_count}")
            print(f"  Total: {leader_count + mature_count}")
            
    except Exception as e:
        print(f"Could not read CSV: {e}")
        
else:
    print("No analysis results available for CSV export - run Test 4 first")

Testing CSV Export with Position Classification
Results exported to standalone_demo_results.csv

First 15 lines of standalone_demo_results.csv:
 1: Donor_Allele,Recipient_Allele,Total_Mismatches,Position,Substitution,Sequence_Region,IMGT_Position
 2: 04:01,05:01,10,64,F>L,Mature,28
 3: 04:01,05:01,10,65,A>V,Mature,29
 4: 04:01,05:01,10,84,A>E,Mature,48
 5: 04:01,05:01,10,113,G>D,Mature,77
 6: 04:01,05:01,10,114,G>E,Mature,78
 7: 04:01,05:01,10,115,P>A,Mature,79
 8: 04:01,05:01,10,116,M>V,Mature,80
 9: 04:01,05:01,10,125,R>K,Mature,89
10: 04:01,05:01,10,199,T>I,Mature,156
11: 04:01,05:01,10,234,V>M,Mature,190
12: 04:01,06:01,15,37,L>V,Mature,8
13: 04:01,06:01,15,38,F>Y,Mature,9
14: 04:01,06:01,15,40,G>L,Mature,11
15: 04:01,06:01,15,65,A>V,Mature,29

Sequence Region Summary:
  Leader mismatches: 0
  Mature mismatches: 50
  Total: 50


## Test 6: Multiple Loci Demonstration

In [11]:
# Test with different loci to show versatility
print("Testing Multiple HLA Loci")
print("=" * 35)

test_loci = ["A", "B", "C", "DRB1"]

for locus in test_loci:
    print(f"\nTesting {locus} locus:")
    
    try:
        # Use HLAtools bridge if available, otherwise fall back to IMGT parsing
        if USE_HLATOOLS:
            alignment = bridge.get_alignment(locus)
        else:
            alignment = load_protein_alignment_enhanced(locus)
        
        if alignment and len(alignment) >= 2:
            alleles = list(alignment.keys())[:2]
            print(f"  Loaded {len(alignment)} alleles")
            print(f"  Sample comparison: {alleles[0]} vs {alleles[1]}")
            
            # Quick comparison
            comp_result = compare_single_alleles(alleles[0], alleles[1], {locus: alignment})
            
            if not comp_result.get('error'):
                print(f"  Result: {comp_result['mismatch_count']} mismatches")
                print(f"  Sequence lengths: {comp_result['donor_sequence_length']} vs {comp_result['recipient_sequence_length']}")
            else:
                print(f"  Comparison failed: {comp_result['error']}")
        else:
            print(f"  Could not load sufficient data for {locus}")
            
    except Exception as e:
        print(f"  Error testing {locus}: {e}")

Testing Multiple HLA Loci

Testing A locus:


  Loaded 8715 alleles
  Sample comparison: A*01:01:01:01 vs A*01:01:01:02N
  Result: 82 mismatches
  Sequence lengths: 365 vs 200

Testing B locus:


  Loaded 10578 alleles
  Sample comparison: B*07:02:01:01 vs B*07:02:01:02
  Result: 0 mismatches
  Sequence lengths: 362 vs 362

Testing C locus:


  Loaded 8807 alleles
  Sample comparison: C*01:02:01:01 vs C*01:02:01:02
  Result: 0 mismatches
  Sequence lengths: 366 vs 366

Testing DRB1 locus:


  Loaded 3856 alleles
  Sample comparison: DRB1*01:01:01:01 vs DRB1*01:01:01:02
  Result: 0 mismatches
  Sequence lengths: 266 vs 266


## Summary

### What This Notebook Demonstrates
- **HLAtools integration**: Alignment data is loaded from the HLAtools R package via rpy2, providing authoritative protein sequences and IMGT position numbering for all major HLA loci
- **Accurate position mapping**: IMGT positions come directly from HLAtools data frame column headers -- no interpolation or text-file parsing heuristics needed
- **Resolved leader/mature boundaries**: The previous discrepancy in leader peptide vs. mature protein classification has been fixed; DPB1 now correctly shows 29 leader and 244 mature positions with a clean -1 to 1 boundary
- **Donor-recipient compatibility analysis**: Full genotype-level comparison across all allele combinations with per-position mismatch detail
- **CSV export**: Results include IMGT position numbers and leader/mature region annotations
- **Multi-locus support**: Tested across A, B, C, DRB1, and DPB1

### Technical Architecture
- `pyard.alignment_bridge.HLAToolsBridge` wraps the HLAtools R package via rpy2
- Alignment data is lazily loaded on first access and cached in-memory
- Falls back to direct IMGT GitHub text-file parsing if HLAtools/rpy2 are unavailable
- All analysis functions accept an `alignment_cache` dict to avoid redundant loads

### Up Next
- Mature protein redux: group alleles that differ only in leader sequences (Issue #354)
- Amino acid mismatch compression: given ambiguous donor and recipient genotypes, determine whether all possible phasing combinations yield the same set of amino acid mismatches